In [ ]:
# Import Essential Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import timedelta
from pandas.plotting import register_matplotlib_converters
from statsmodels.tsa.statespace.sarimax import SARIMAX
register_matplotlib_converters()
from time import time

# Prepare Bicycle Dataset
Use dtheday as the date and cnt as total daily rentals. One row is one day across the recorded system; cnt is not the number of unique people or bikes needed at a station.
- Parse and sort dates.
- Check for duplicate dates.
- Check for missing calendar days.
- Check for missing count values.
- Check for negative or non-integer counts.

Record each findings. The unmodified daily file should contain 731 dates from January 1 2011 to December 31 2012. **If checks fails fail, do not invent observations or silently delete dates.** Use only the dated ctn series as the SARIMA input. Other fields, including casual, registered, and measured weather, are not model inputs. You do not need to create lag_1 or lag_7 feature columns; SARIMA handles its own time dependence.

> Inspecting the dataset.

In [90]:
# The program below loads data from day.csv. It parses the date to be in day/month/Year format. Sort the data dteday.
# Marking the the dteday as the index column unlocks pandas' built in time-series-engine,
# which makes queries and plotting easier.
bike_shares = pd.read_csv("../data/raw/day.csv",parse_dates=[1], index_col=[1], date_format="%d/%m/%Y").sort_values(by="dteday")
bike_shares.head()

,instant,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
dteday,,,,,,,,,,,,,,,
2011-01-01,1,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
2011-01-02,2,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2011-01-03,3,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
2011-01-04,4,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
2011-01-05,5,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [95]:
# Checking for duplicate dates.
assert bike_shares.index.duplicated().sum() == 0, "Data set contains duplicated dates."
if bike_shares.index.duplicated().sum() > 0:
    print(bike_shares[bike_shares.index.duplicated()])
else:
    print("No duplicate dates found.")

# Check for missing calendar days.
#January 1 2011 to December 31 2012
start_date = datetime(2011, 1, 1)
end_date = datetime(2012, 12, 31)
missing_dates = pd.date_range(start=start_date, end=end_date).difference(pd.to_datetime(bike_shares.index))
assert len(missing_dates) == 0, "Data set contains missing dates."
if len(missing_dates) > 0:
    print(missing_dates)
else:
    print("No missing dates found")

# Check for missing count values.
assert bike_shares.cnt.isna().sum() == 0, "Data set contains missing value."
if bike_shares.cnt.isna().sum() > 0:
    print(bike_shares[bike_shares.cnt.isna()])
else:
    print("No missing count values found")

# Check for negative or non-integer counts
# First, converts all ctn records to numeric (e.g. int64 or float64).
# It ensures that strings are converted to proper numerical data types
# and non numerical data is converted into NaN.
invalid_mask =  (bike_shares.cnt < 0) | ((bike_shares.cnt % 1) != 0)
assert not invalid_mask.any(), "Data set contains negative or non-integer values"
if invalid_mask.any():
    print(bike_shares[invalid_mask])
else:
    print("No negative or non-integer values found.")

No duplicate dates found.
No missing dates found
No missing count values found
No negative or non-integer values found.


Based on the outputs, the bike_shares data set is clean. It does not contain:
- any duplicated dates, 
- missing dates, 
- missing daily rental records, and 
- invalid daily rental records.

> Exploring the dataset.